# StudyMate AI — Intelligent Lecture Notes Analyzer
**Student:** Rameen Muzammil Ejaz  
**Registration No.:** FA24-BSE-160  
**Course:** Machine Learning Fundamentals  
**Project:** End-to-End Machine Learning Systems Development using Unstructured Data

**Deployment URL:** Add your live Hugging Face Space URL here after deployment.

## Objective
Extract features from unstructured text, compare classical ML classifiers, use pretrained Transformers for summarization and study-aid generation, evaluate summaries with ROUGE, and deploy an interactive application.

**Dataset note:** AG News has four broad topics (World, Sports, Business, Sci/Tech). The classifier predicts broad text topics, not university course subjects. The summarization and generation components operate on uploaded lecture notes.

## 1. Install dependencies

In [ ]:
!pip -q install datasets transformers evaluate rouge-score gradio pymupdf joblib scikit-learn seaborn

## 2. Import libraries and configure experiment

In [ ]:
import os, re, json, random, warnings, joblib
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score
from transformers import pipeline
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings("ignore")
USE_FULL_TRAINING = False  # Set True to train on all 120,000 training records
SAMPLE_PER_CLASS = 8000   # Used only when USE_FULL_TRAINING=False
MAX_FEATURES = 30000
DEVICE = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if DEVICE == 0 else "CPU")

## 3. Download the full classification dataset

In [ ]:
dataset = load_dataset("fancyzhx/ag_news")
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])
label_names = ["World", "Sports", "Business", "Sci/Tech"]
print("Full training rows:", len(train_df), "| Full test rows:", len(test_df))
display(train_df.head())
display(train_df.label.value_counts().sort_index().rename(index=dict(enumerate(label_names))))

## 4. Exploratory data analysis

In [ ]:
train_df["word_count"] = train_df.text.astype(str).str.split().str.len()
plt.figure(figsize=(8,4))
sns.countplot(data=train_df, x="label")
plt.xticks(range(4), label_names); plt.title("AG News class distribution"); plt.show()
plt.figure(figsize=(8,4))
sns.histplot(train_df.word_count, bins=40); plt.title("Article word-count distribution"); plt.show()

## 5. Text preprocessing

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

if USE_FULL_TRAINING:
    experiment_train = train_df.copy()
else:
    experiment_train = (train_df.groupby("label", group_keys=False)
                        .apply(lambda x: x.sample(n=min(len(x), SAMPLE_PER_CLASS), random_state=SEED))
                        .reset_index(drop=True))
X_train = experiment_train.text.map(clean_text)
y_train = experiment_train.label.astype(int)
X_test = test_df.text.map(clean_text)
y_test = test_df.label.astype(int)
print("Training records used:", len(X_train), "| Held-out test records:", len(X_test))

## 6. Feature extraction using TF-IDF
TF-IDF converts unstructured text into numerical feature vectors. Word unigrams and bigrams are used.

In [ ]:
tfidf_preview = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1,2), min_df=2, sublinear_tf=True)
X_train_tfidf = tfidf_preview.fit_transform(X_train)
X_test_tfidf = tfidf_preview.transform(X_test)
print("Train matrix:", X_train_tfidf.shape, "| Test matrix:", X_test_tfidf.shape)
print("Sample vocabulary:", list(tfidf_preview.vocabulary_)[:20])

## 7. Train and compare five ML algorithms

In [ ]:
models = {
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.5),
    "Logistic Regression": LogisticRegression(max_iter=1000, C=4.0),
    "Linear SVM": LinearSVC(C=1.0),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_features="sqrt", n_jobs=-1, random_state=SEED),
    "k-NN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
}
results, fitted, predictions = [], {}, {}
for name, algorithm in models.items():
    print("Training:", name)
    pipe = Pipeline([("tfidf", TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1,2), min_df=2, sublinear_tf=True)),
                     ("classifier", algorithm)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average="macro", zero_division=0)
    results.append({"Model":name, "Accuracy":accuracy_score(y_test,pred),
                    "Macro Precision":p, "Macro Recall":r, "Macro F1":f1})
    fitted[name], predictions[name] = pipe, pred
results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
display(results_df)
best_name = results_df.iloc[0]["Model"]
best_model = fitted[best_name]
best_predictions = predictions[best_name]
print("Best model:", best_name)
print(classification_report(y_test, best_predictions, target_names=label_names, zero_division=0))

## 8. Model comparison visualization

In [ ]:
results_df.set_index("Model")[["Accuracy","Macro Precision","Macro Recall","Macro F1"]].plot(kind="bar", figsize=(12,5), ylim=(0,1))
plt.title("Classical ML Model Comparison"); plt.ylabel("Score"); plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.show()

## 9. Confusion matrix

In [ ]:
plt.figure(figsize=(7,6))
sns.heatmap(confusion_matrix(y_test,best_predictions), annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Best Model Confusion Matrix"); plt.show()

## 10. 10-fold cross-validation
For runtime, CV uses a stratified subset and three representative algorithms. The independent held-out test set remains the primary comparison.

In [ ]:
CV_SAMPLE_PER_CLASS = 1000
cv_df = (experiment_train.groupby("label", group_keys=False)
         .apply(lambda x: x.sample(n=min(len(x),CV_SAMPLE_PER_CLASS), random_state=SEED))
         .reset_index(drop=True))
X_cv, y_cv = cv_df.text.map(clean_text), cv_df.label.astype(int)
cv_algorithms = {"Naive Bayes":MultinomialNB(alpha=.5),
                 "Logistic Regression":LogisticRegression(max_iter=1000,C=4.0),
                 "Linear SVM":LinearSVC(C=1.0)}
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
cv_rows = []
for name, clf in cv_algorithms.items():
    pipe = Pipeline([("tfidf",TfidfVectorizer(max_features=MAX_FEATURES,ngram_range=(1,2),min_df=2)),
                     ("classifier",clf)])
    scores = cross_val_score(pipe,X_cv,y_cv,cv=skf,scoring="accuracy",n_jobs=-1)
    cv_rows.append({"Model":name,"Mean CV Accuracy":scores.mean(),"Std CV Accuracy":scores.std()})
cv_results = pd.DataFrame(cv_rows).sort_values("Mean CV Accuracy",ascending=False)
display(cv_results)

## 11. Save the best model and metadata

In [ ]:
os.makedirs("artifacts",exist_ok=True)
joblib.dump(best_model,"artifacts/best_classifier.joblib")
joblib.dump(best_model.named_steps["tfidf"],"artifacts/tfidf_vectorizer.joblib")
results_df.to_csv("artifacts/model_comparison.csv",index=False)
with open("artifacts/model_metadata.json","w") as f:
    json.dump({"best_model":best_name,"labels":label_names,
               "training_records_used":len(X_train),"test_records":len(X_test),
               "dataset":"fancyzhx/ag_news","results":results_df.to_dict(orient="records")},f,indent=2)
print(os.listdir("artifacts"))

## 12. Test topic classification on new text

In [ ]:
def predict_topic(text):
    return label_names[int(best_model.predict([clean_text(text)])[0])]
sample = "The new processor improves computing performance and introduces a faster graphics architecture."
print("Predicted topic:",predict_topic(sample))

## 13. Pretrained Transformer: summarization with DistilBART

In [ ]:
summarizer = pipeline("summarization",model="sshleifer/distilbart-cnn-12-6",device=DEVICE)
def summarize_text(text,max_length=100,min_length=20):
    words=text.split(); chunks=[]; current=[]
    for word in words:
        current.append(word)
        if len(current)>=550:
            chunks.append(" ".join(current)); current=[]
    if current: chunks.append(" ".join(current))
    partial=[summarizer(c,max_length=max_length,min_length=min_length,do_sample=False)[0]["summary_text"]
             for c in chunks[:10]]
    if len(partial)>1:
        return summarizer(" ".join(partial),max_length=max_length,min_length=min_length,do_sample=False)[0]["summary_text"]
    return partial[0]
demo_text = ("Machine learning is a field of artificial intelligence that enables computer systems to learn patterns from data. "
             "Supervised learning uses labeled examples to train predictive models. Unsupervised learning discovers patterns "
             "without labels. Natural language processing allows computers to analyze and generate human language. "
             "Transformer models use attention mechanisms to process contextual relationships between words.")
print(summarize_text(demo_text))

## 14. Summarization evaluation using ROUGE
The SAMSum validation set supplies human-written reference summaries. A fixed sample is evaluated to control runtime; this is separate from the full AG News classification dataset.

In [ ]:
import evaluate
rouge = evaluate.load("rouge")
summary_ds = load_dataset("knkarthick/samsum",split="validation")
N_ROUGE = 100
eval_rows = summary_ds.select(range(min(N_ROUGE,len(summary_ds))))
generated, references = [], []
for row in eval_rows:
    generated.append(summarize_text(row["dialogue"],max_length=80,min_length=15))
    references.append(row["summary"])
rouge_scores = rouge.compute(predictions=generated,references=references,use_stemmer=True)
for metric,score in rouge_scores.items():
    print(f"{metric}: {score:.4f}")

## 15. Pretrained instruction-tuned Transformer: FLAN-T5 for flashcards and MCQs

In [ ]:
study_generator = pipeline("text2text-generation",model="google/flan-t5-base",device=DEVICE)
def generate_flashcards(text):
    prompt = "Create 6 concise study flashcards. Format each as Q: question and A: answer.\n\n" + text[:3000]
    return study_generator(prompt,max_new_tokens=400,do_sample=False)[0]["generated_text"]
def generate_mcqs(text):
    prompt = "Create 5 factual multiple-choice questions with four options A-D and mark the correct answer.\n\n" + text[:3000]
    return study_generator(prompt,max_new_tokens=450,do_sample=False)[0]["generated_text"]
print("FLASHCARDS\n",generate_flashcards(demo_text))
print("\nMCQS\n",generate_mcqs(demo_text))

## 16. Download artifacts for deployment

In [ ]:
import shutil
shutil.make_archive("StudyMate_AI_artifacts","zip","artifacts")
from google.colab import files
files.download("StudyMate_AI_artifacts.zip")

## 17. Hugging Face Spaces deployment
1. Create a new Hugging Face Space and select Gradio.
2. Upload `app.py` and `requirements.txt`.
3. Upload `best_classifier.joblib`, `tfidf_vectorizer.joblib`, and `model_metadata.json` inside an `artifacts/` folder.
4. Wait for dependencies and pretrained models to download.
5. Copy the public Space URL to the Deployment URL line at the top of this notebook.

## 18. Conclusion
StudyMate AI demonstrates unstructured-text extraction, preprocessing, TF-IDF feature representation, classical ML comparison, evaluation, pretrained Transformer inference, ROUGE evaluation and cloud deployment.

Before submission, run all cells, use actual measured scores, test the deployed app, add the live URL and include screenshots of the application and results.